In [ ]:
# Importar as bibliotecas necessárias

import json
import os
import pandas as pd
from pathlib import Path
import sys

In [ ]:
# Resolver os 'imports' do projeto 

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

import utils.functions.date_transform as date_transform
import utils.functions.tax_calculation as tax_calculation
from   utils.classes.pandas_dataframe import PandasDataframe as pd_dataframe
import errors_redeem as errors

In [ ]:
# Traz as opções de configuração

json_file = os.path.abspath('../../options.json')

with open(json_file, 'r') as j_file:
    json_data = json.load(j_file)

In [ ]:
# Declaração dos caminhos

dir_data = os.path.join(os.path.abspath('../../'), 'data')
dir_cvm  = os.path.join(dir_data, 'cvm')
dir_processed  = os.path.join(dir_cvm, 'processed')

In [ ]:
# Manejo dos dados do investimento

INVESTMENT_CNPJ = json_data["REDEEM"]["INVESTMENT"]
INVESTMENT_DATA  = json_data["INVESTMENTS"][INVESTMENT_CNPJ]

In [ ]:
# Cálculo de impostos

sum_days = date_transform.calculate_days(INVESTMENT_DATA['HistoricoCompraData'], INVESTMENT_DATA['HistoricoVendaData'])
income_percentage = tax_calculation.income_tax_calculation(sum_days) 
iof_percentage = tax_calculation.iof_tax_calculation(sum_days) 

In [ ]:
# Recolher dados do investimento

INVESTMENT_ARCHIVE_NAME = "_".join(INVESTMENT_DATA['Fundo'].split(' ')) + '.csv'
INVESTMENT_PATH_ARCHIVE = os.path.join(dir_processed, INVESTMENT_ARCHIVE_NAME)

investment_df = pd_dataframe(INVESTMENT_PATH_ARCHIVE, None)
investment_df.csv_to_df()

In [ ]:
# Ajuste das datas, do fim de semana, para semana

date_investment_start = INVESTMENT_DATA['HistoricoCompraData']
date_investment_end   = INVESTMENT_DATA['HistoricoVendaData']

date_investment_start = date_transform.transform_string_into_date(date_investment_start)
date_investment_end   = date_transform.transform_string_into_date(date_investment_end)

date_of_week_investment_start = date_investment_start.weekday()
date_of_week_investment_end   = date_investment_end.weekday()

if (date_of_week_investment_start == 5 or date_of_week_investment_start == 6):
    DAYS_IN_WEEK = 7
    print('The date "buyed" was on the weekend, because of that, date was adjusted to monday')
    date_investment_start = date_transform.add_days_on_date(date_investment_start, DAYS_IN_WEEK - date_of_week_investment_start)

if date_of_week_investment_end == 5 or date_of_week_investment_end == 6:
    DAYS_IN_WEEK = 7
    print('The date "sold" was on the weekend, because of that, date was adjusted to monday')
    date_investment_end = date_transform.add_days_on_date(date_investment_end, DAYS_IN_WEEK - date_of_week_investment_end)

In [ ]:
# Preparação das variáveis imutáveis para validação das datas

FIRST_ROW_DATE = investment_df.df.iloc[0].DT_COMPTC
LAST_ROW_DATE  = investment_df.df.iloc[-1].DT_COMPTC

DAYS_OFFSET = json_data["REDEEM"]["DAYS_OFFSET"]
FIRST_ROW_INDEX = 0

In [ ]:
# Validação dos dados processados do investimento e da compra no aplicativo

index_purchase_investment = 0
str_investment_start  = date_transform.transform_date_into_string(date_investment_start)
# 'date_investment_start'

str_first_row  = FIRST_ROW_DATE
date_first_row = date_transform.transform_string_into_date(str_first_row)

is_date_buyed_before_date_first_row = date_investment_start < date_first_row

if is_date_buyed_before_date_first_row:
    date_new_investment_start = date_transform.add_days_on_date(date_investment_start, DAYS_OFFSET)

    if date_new_investment_start < date_first_row:
        error                 = errors.Errors_Redeem_Simulation()
        dto_invalid_date_init = error.dto_invalid_date_init_before_row_start(str_investment_start, date_new_investment_start, date_first_row, DAYS_OFFSET)
        error_message         = error.str_invalid_date_init_before_row_start(**dto_invalid_date_init)

        raise Exception(error_message)
        
    # O primeiro registro da 'df' é igual ao primeiro dia de compra
    index_purchase_investment = FIRST_ROW_INDEX
    date_investment_start = date_first_row
    str_investment_start  = str_first_row
else:
    is_valid, index_purchase_investment = investment_df.find_row_date_greater_or_equals_than_indicated(str_investment_start)

    if not is_valid:
        error = errors.Errors_Redeem_Simulation()
        dto_invalid_date_init = error.dto_invalid_date_init_after_row_end(str_investment_start, LAST_ROW_DATE)
        error_message         = error.str_invalid_date_init_after_row_end(**dto_invalid_date_init)

        raise Exception(error_message)

    str_investment_start = investment_df.df.iloc[index_purchase_investment].DT_COMPTC
    date_investment_start = date_transform.transform_string_into_date(str_investment_start)

In [ ]:
# Validação da data de fim do investimento

first_row_date = date_transform.transform_string_into_date(FIRST_ROW_DATE)
last_row_date  = date_transform.transform_string_into_date(LAST_ROW_DATE)

str_investment_end = date_transform.transform_date_into_string(date_investment_end)

# Ajuste de venda com dias válidos na planilha da CVM
is_valid, index_sold_investment = investment_df.find_row_date_greater_or_equals_than_indicated(str_investment_end)
str_investment_end  = investment_df.df.iloc[index_sold_investment].DT_COMPTC
date_investment_end = date_transform.transform_string_into_date(str_investment_end)

if not is_valid:
    error = errors.Errors_Redeem_Simulation()
    dto_invalid_date_init = error.dto_invalid_date_init_after_row_end(str_investment_start, LAST_ROW_DATE)
    error_message         = error.str_invalid_date_init_after_row_end(**dto_invalid_date_init)

if first_row_date > date_investment_end:
    error                = errors.Errors_Redeem_Simulation()
    dto_invalid_date_end = error.dto_invalid_date_end_before_row_start(str_investment_end, first_row_date)
    error_message        = error.str_invalid_date_end_before_row_start(**dto_invalid_date_end)

    raise Exception(error_message)
    
if last_row_date < date_investment_end:
    error                = errors.Errors_Redeem_Simulation()
    dto_invalid_date_end = error.dto_invalid_date_end_after_row_end(str_investment_end, last_row_date)
    error_message        = error.str_invalid_date_end_after_row_end(**dto_invalid_date_end)
    
    raise Exception(error_message)

In [ ]:
# Aplicação das cotas sobre o investimento

APPLICATION_QUOTA = INVESTMENT_DATA['CotaApplicacao']
REDEEM_QUOTA      = INVESTMENT_DATA['CotaResgate']

try:
    investment_start_quota      = investment_df.df.iloc[index_purchase_investment + APPLICATION_QUOTA]
    str_investment_start_quota  = investment_start_quota.DT_COMPTC
    date_investment_start_quota = date_transform.transform_string_into_date(str_investment_start_quota)
except:
    error                 = errors.Errors_Redeem_Simulation()
    dto_invalid_date_init = error.dto_invalid_date_init_quota_adjust(str_investment_start_quota, last_row_date, APPLICATION_QUOTA)
    error_message         = error.str_invalid_date_init_quota_adjust(**dto_invalid_date_init)

    raise Exception(error_message)

try:
    investment_end_quota      = investment_df.df.iloc[index_sold_investment + REDEEM_QUOTA]
    str_investment_end_quota  = investment_start_quota.DT_COMPTC
    date_investment_end_quota = date_transform.transform_string_into_date(str_investment_start_quota)
except:
    error                = errors.Errors_Redeem_Simulation()
    dto_invalid_date_end = error.dto_invalid_date_end_quota_adjust(str_investment_end, last_row_date, REDEEM_QUOTA)
    error_message        = error.str_invalid_date_end_quota_adjust(**dto_invalid_date_end)

    raise Exception(error_message)

In [ ]:
# Validação de cotação (data de início + cota >= a data de fim do investimento)

if date_investment_start_quota > date_investment_end:
    error                       = errors.Errors_Redeem_Simulation()
    dto_conflict_date_start_end = error.dto_conflict_date_start_end(str_investment_start_quota, str_investment_end)
    error_message               = error.str_conflict_date_start_end(**dto_conflict_date_start_end)

    raise Exception(error_message)

In [ ]:
# Fazer a simulação do investimento

QUOTA_QUANTITY = json_data['INVESTMENTS'][json_data['REDEEM']['INVESTMENT']]['Cotas']

INVESTMENT_VALUE_START = QUOTA_QUANTITY * investment_start_quota.VL_QUOTA
INVESTMENT_VALUE_END   = QUOTA_QUANTITY * investment_end_quota.VL_QUOTA

INVESTMENT_VALUE_GROSS = INVESTMENT_VALUE_END

[investment_value_net, income_tax_value, iof_tax_value] = [INVESTMENT_VALUE_END, 0, 0]

if INVESTMENT_VALUE_START < INVESTMENT_VALUE_END:
    profit = INVESTMENT_VALUE_END - INVESTMENT_VALUE_START
    
    income_tax_value   = profit * (income_percentage / 100)
    profit_with_taxes  = profit - income_tax_value

    iof_tax_value      = profit_with_taxes * (iof_percentage / 100)
    profit_with_taxes  = profit_with_taxes - iof_tax_value

    investment_value_net = INVESTMENT_VALUE_START + profit_with_taxes

In [ ]:
# Criar display dos dados

investment_simulation = [
    {
        'Investment Start':       INVESTMENT_VALUE_START,
        'Investment End (Gross)': INVESTMENT_VALUE_GROSS,
        'Investment End (Net)':   investment_value_net,
        'iof_tax_value':          iof_tax_value,
        'income_tax_value':       income_tax_value,
    }
]

df = pd.DataFrame(investment_simulation)
print(df.to_string(index=False))